In [ ]:
import torch
import torch.nn as nn

In [ ]:
class YOLOv1(nn.Module):
    def __init__(self, num_classes, grid_size=7, boxes_per_cell=2):
        super().__init__()
        self.num_classes = num_classes
        self.grid_size = grid_size
        self.boxes_per_cell = boxes_per_cell
        if grid_size != 7:
            raise ValueError("This original fully connected head expects a 7x7 grid and 448x448 images.")
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=(7,7), stride=2, padding=3)

        self.conv2 = nn.Conv2d(in_channels=64, out_channels=192, kernel_size=(3,3), stride=1, padding=1)

        self.conv3_1 = nn.Conv2d(in_channels=192, out_channels=128, kernel_size=(1,1), stride=1, padding=0)
        self.conv3_2 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3,3), stride=1, padding=1)
        self.conv3_3 = nn.Conv2d(in_channels=256, out_channels=256, kernel_size=(1,1), stride=1, padding=0)
        self.conv3_4 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), stride=1, padding=1)

        self.conv4_1_1 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=(1,1), stride=1, padding=0)
        self.conv4_1_2 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), stride=1, padding=1)
        self.conv4_2_1 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=(1,1), stride=1, padding=0)
        self.conv4_2_2 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), stride=1, padding=1)
        self.conv4_3_1 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=(1,1), stride=1, padding=0)
        self.conv4_3_2 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), stride=1, padding=1)
        self.conv4_4_1 = nn.Conv2d(in_channels=512, out_channels=256, kernel_size=(1,1), stride=1, padding=0)
        self.conv4_4_2 = nn.Conv2d(in_channels=256, out_channels=512, kernel_size=(3,3), stride=1, padding=1)
        self.conv4_5 = nn.Conv2d(in_channels=512, out_channels=512, kernel_size=(1,1), stride=1, padding=0)
        self.conv4_6 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)

        self.conv5_1_1 = nn.Conv2d(in_channels=1024, out_channels=512, kernel_size=(1,1), stride=1, padding=0)
        self.conv5_1_2 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)
        self.conv5_2_1 = nn.Conv2d(in_channels=1024, out_channels=512, kernel_size=(1,1), stride=1, padding=0)
        self.conv5_2_2 = nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)
        self.conv5_3 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)
        self.conv5_4 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3), stride=2, padding=1)

        self.conv6_1 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)
        self.conv6_2 = nn.Conv2d(in_channels=1024, out_channels=1024, kernel_size=(3,3), stride=1, padding=1)

        self.fc1 = nn.Linear(in_features=50176, out_features=4096)
        self.fc2 = nn.Linear(4096, grid_size * grid_size * (boxes_per_cell * 5 + num_classes))

        self.dropout = nn.Dropout(p=0.5)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self,x):
        x = self.conv1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.maxpool(x)
        x = self.conv3_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv3_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv3_3(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv3_4(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.maxpool(x)
        x = self.conv4_1_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_1_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_2_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_2_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_3_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_3_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_4_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_4_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_5(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv4_6(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.maxpool(x)
        x = self.conv5_1_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv5_1_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv5_2_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv5_2_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv5_3(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv5_4(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv6_1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.conv6_2(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = nn.LeakyReLU(negative_slope=0.1)(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x.reshape(x.shape[0], self.grid_size, self.grid_size, self.boxes_per_cell * 5 + self.num_classes)
